In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

ModuleNotFoundError: No module named 'kagglehub'

# 1. Khai báo dataset

In [ ]:
import pandas as pd
from pathlib import Path

root = Path("/kaggle/input/competitions/edge-ai-challenge-2026-lan-2")

train = pd.read_csv(root / "intent_train.csv")
test  = pd.read_csv(root / "intent_test.csv")
sub   = pd.read_csv(root / "sample_submission.csv")

train_imgs = sorted((root / "train/images").glob("*.png"))
train_lbls = sorted((root / "train/labels").glob("*.txt"))
test_imgs  = sorted((root / "test/images").glob("*.png"))

print(len(train_imgs), len(train_lbls), len(test_imgs))

img_ids = {p.stem for p in train_imgs}
lbl_ids = {p.stem for p in train_lbls}
print("Ảnh thiếu label:", len(img_ids - lbl_ids))
print("Label thiếu ảnh:", len(lbl_ids - img_ids))

FileNotFoundError: [Errno 2] No such file or directory: '\\kaggle\\input\\competitions\\edge-ai-challenge-2026-lan-2\\intent_train.csv'

In [ ]:
print(train.shape, test.shape, sub.shape)
print(train.head())
print(test.head())
print(sub.head())

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

img = Image.open(train_imgs[0])
print(img.size, img.mode)
plt.imshow(img); plt.axis("off"); plt.show()

# 2. Đọc danh sách class

In [ ]:
with open(root / "classes.txt", encoding="utf-8") as f:
    classes = [line.strip() for line in f if line.strip()]

print("Số lớp:", len(classes))
print(classes)

Gán số từ 0 toeis 9

In [ ]:
id2name = {i: name for i, name in enumerate(classes)}
name2id = {name: i for i, name in enumerate(classes)}

print(id2name)

In [ ]:
import numpy as np

all_ids = []
for p in train_lbls:
    arr = np.loadtxt(p, ndmin=2)
    if arr.size:                      # bỏ qua file label rỗng
        all_ids.extend(arr[:, 0].astype(int))

all_ids = np.array(all_ids)
print("id nhỏ nhất / lớn nhất:", all_ids.min(), all_ids.max())

# đếm số box mỗi lớp
from collections import Counter
cnt = Counter(all_ids)
for i in sorted(cnt):
    print(f"{i:3d} {id2name[i]:20s} {cnt[i]}")

# 3. Check dataset

check 3 file csv

In [ ]:
print(train.shape, test.shape, sub.shape)
display(train.head())
display(test.head())
display(sub.head())

print("Giá trị thiếu (train):\n", train.isnull().sum())
print("Dòng trùng (train):", train.duplicated().sum())
print("Cột train:", train.columns.tolist())
print("Cột test :", test.columns.tolist())
print("Cột sub  :", sub.columns.tolist())

check file labels

In [ ]:
import numpy as np

bad = []
empty = []
n_boxes = {}

for p in train_lbls:
    lines = [l.split() for l in open(p) if l.strip()]
    if not lines:
        empty.append(p.stem)
        continue
    n_boxes[p.stem] = len(lines)
    for parts in lines:
        if len(parts) != 5:
            bad.append((p.stem, "sai số cột", parts)); continue
        c, x, y, w, h = int(parts[0]), *map(float, parts[1:])
        if not (0 <= c < len(classes)):
            bad.append((p.stem, "class_id ngoài khoảng", c))
        if not all(0 <= v <= 1 for v in (x, y, w, h)):
            bad.append((p.stem, "toạ độ ngoài [0,1]", (x, y, w, h)))
        if w <= 0 or h <= 0:
            bad.append((p.stem, "box rỗng", (w, h)))

print("File label rỗng:", len(empty))
print("Dòng lỗi:", len(bad))
for b in bad[:10]:
    print(b)
print("Số box/ảnh:", pd.Series(n_boxes).describe())

check img 

In [ ]:
from PIL import Image
from collections import Counter

sizes, modes, corrupt = Counter(), Counter(), []

for p in train_imgs + test_imgs:
    try:
        with Image.open(p) as im:
            im.verify()
        with Image.open(p) as im:
            sizes[im.size] += 1
            modes[im.mode] += 1
    except Exception as e:
        corrupt.append((p.name, str(e)))

print("Ảnh hỏng:", len(corrupt), corrupt[:5])
print("Kích thước:", sizes.most_common(5))
print("Mode:", modes)

check img + label

In [ ]:
test_ids = [p.stem for p in test_imgs]
print("Ví dụ id test:", test_ids[:9])
print("Số ảnh test:", len(test_ids), "| số dòng sub:", len(sub))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def show(idx, ax):
    p = train_imgs[idx]
    img = Image.open(p)
    W, H = img.size
    ax.imshow(img); ax.axis("off"); ax.set_title(p.stem)
    lbl = root / "train/labels" / f"{p.stem}.txt"
    for line in open(lbl):
        c, x, y, w, h = map(float, line.split())
        ax.add_patch(patches.Rectangle(((x-w/2)*W, (y-h/2)*H), w*W, h*H,
                                       fill=False, edgecolor="red", lw=2))
        ax.text((x-w/2)*W, (y-h/2)*H, classes[int(c)], color="white",
                bbox=dict(facecolor="red", alpha=0.6), fontsize=8)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, i in zip(axes.ravel(), np.random.choice(len(train_imgs), 6, replace=False)):
    show(i, ax)
plt.tight_layout(); plt.show()

# 4. Check YOLO

In [ ]:
import numpy as np
import pandas as pd
from collections import Counter

with open(root / "classes.txt", encoding="utf-8") as f:
    classes = [l.strip() for l in f if l.strip()]
nc = len(classes)

rows, errors, empty = [], [], []

for p in train_lbls:
    lines = [l.strip() for l in open(p) if l.strip()]
    if not lines:
        empty.append(p.stem)
        continue
    for ln, line in enumerate(lines, 1):
        parts = line.split()
        if len(parts) != 5:
            errors.append((p.stem, ln, "sai số cột", len(parts)))
            continue
        try:
            c = int(float(parts[0]))
            x, y, w, h = map(float, parts[1:])
        except ValueError:
            errors.append((p.stem, ln, "không phải số", line))
            continue
        if not (0 <= c < nc):
            errors.append((p.stem, ln, "class_id ngoài khoảng", c))
        if not (0 <= x <= 1 and 0 <= y <= 1):
            errors.append((p.stem, ln, "tâm ngoài [0,1]", (x, y)))
        if not (0 < w <= 1 and 0 < h <= 1):
            errors.append((p.stem, ln, "w/h không hợp lệ", (w, h)))
        if x - w/2 < -1e-3 or x + w/2 > 1 + 1e-3 or y - h/2 < -1e-3 or y + h/2 > 1 + 1e-3:
            errors.append((p.stem, ln, "box tràn ra ngoài ảnh", (x, y, w, h)))
        rows.append([p.stem, c, x, y, w, h])

df = pd.DataFrame(rows, columns=["id", "cls", "x", "y", "w", "h"])
df["name"] = df["cls"].map(lambda i: classes[i] if 0 <= i < nc else "?")
df["area"] = df["w"] * df["h"]

print("Số lớp:", nc)
print("Tổng số box:", len(df), "| số ảnh có box:", df["id"].nunique())
print("File label rỗng:", len(empty), empty[:5])
print("Số dòng lỗi:", len(errors))
for e in errors[:10]:
    print("  ", e)

In [ ]:
dup = df.duplicated(subset=["id", "cls", "x", "y", "w", "h"]).sum()
print("Box trùng hệt nhau:", dup)

In [ ]:
cnt = df["name"].value_counts()
print(cnt)

import matplotlib.pyplot as plt
cnt.plot(kind="bar", figsize=(10, 4), title="Số box mỗi lớp")
plt.show()

print("Lớp không có box nào:", [c for c in classes if c not in set(df["name"])])

In [ ]:
print(df.groupby("id").size().describe())      # số object mỗi ảnh
print(df[["w", "h", "area"]].describe())        # kích thước box (chuẩn hoá)

print("Box rất nhỏ (area < 0.001):", (df["area"] < 0.001).sum())

In [ ]:
img_ids = {p.stem for p in train_imgs}
print("Ảnh không có dòng box nào:", len(img_ids - set(df["id"])))

In [ ]:
big = df[df["w"] > 0.5]
print(big)
print(big["name"].value_counts())

In [ ]:
zc = df[df["name"] == "zebra_crossing"]
print(zc[["w", "h", "area"]].describe())
print("Box có w >= 0.99:", (zc["w"] >= 0.99).sum())

# xem có box nào tràn ra ngoài ảnh không
print("Lệch trái/phải:", ((zc["x"] - zc["w"]/2) < -1e-3).sum(), ((zc["x"] + zc["w"]/2) > 1 + 1e-3).sum())

In [ ]:
print(df["name"].value_counts())
print("Số dòng lỗi:", len(errors), "| label rỗng:", len(empty))
print(Image.open(train_imgs[0]).size)

# 5. Trực quan hóa 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

cmap = plt.get_cmap("tab10")
colors = {i: cmap(i) for i in range(len(classes))}

def show(idx, ax):
    p = train_imgs[idx]
    img = Image.open(p).convert("RGB")
    W, H = img.size
    ax.imshow(img); ax.axis("off"); ax.set_title(p.stem)
    for line in open(root / "train/labels" / f"{p.stem}.txt"):
        c, x, y, w, h = map(float, line.split())
        c = int(c)
        x1, y1 = (x - w/2) * W, (y - h/2) * H
        ax.add_patch(patches.Rectangle((x1, y1), w*W, h*H, fill=False,
                                       edgecolor=colors[c], lw=2))
        ax.text(x1, y1, classes[c], color="white", fontsize=8,
                bbox=dict(facecolor=colors[c], alpha=0.7, pad=1))

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
for ax, i in zip(axes.ravel(), np.random.choice(len(train_imgs), 6, replace=False)):
    show(i, ax)
plt.tight_layout(); plt.show()

In [ ]:
first_per_class = df.groupby("cls")["id"].first()
ids = {p.stem: i for i, p in enumerate(train_imgs)}

fig, axes = plt.subplots(2, 5, figsize=(24, 8))
for ax, (c, img_id) in zip(axes.ravel(), first_per_class.items()):
    show(ids[img_id], ax)
    ax.set_title(f"{classes[c]} ({img_id})")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

df["name"].value_counts().plot(kind="bar", ax=axes[0], title="Số box mỗi lớp")
df.groupby("id").size().plot(kind="hist", bins=12, ax=axes[1], title="Số box mỗi ảnh")
axes[2].scatter(df["w"], df["h"], c=df["cls"].map(lambda i: colors[i]), s=12)
axes[2].set_xlabel("w"); axes[2].set_ylabel("h"); axes[2].set_title("Kích thước box (chuẩn hoá)")

plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.hist2d(df["x"], 1 - df["y"], bins=30, cmap="viridis")
plt.colorbar(label="số box"); plt.title("Mật độ tâm box"); plt.show()

# 6. Chia train/val ttheo class 

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

cls_count = df["cls"].value_counts()
img_cls = (df.groupby("id")["cls"]
             .apply(lambda s: min(set(s), key=lambda c: cls_count[c])))

ids = img_cls.index.to_numpy()
strat = img_cls.to_numpy()

# nhóm nào < 2 ảnh thì không stratify được -> đưa thẳng vào train
vc = pd.Series(strat).value_counts()
rare = set(vc[vc < 2].index)
mask = np.array([s in rare for s in strat])

ids_rare = ids[mask]
ids_ok, strat_ok = ids[~mask], strat[~mask]

train_ids, val_ids = train_test_split(
    ids_ok, test_size=0.2, random_state=42, stratify=strat_ok
)
train_ids = list(train_ids) + list(ids_rare)

no_box = sorted({p.stem for p in train_imgs} - set(ids))
train_ids = train_ids + no_box

print("Train:", len(train_ids), "| Val:", len(val_ids))
print("Ảnh của nhóm hiếm đưa vào train:", len(ids_rare))

In [ ]:
train_ids, val_ids = train_test_split(ids, test_size=0.2, random_state=42)

In [ ]:
d_tr = df[df["id"].isin(train_ids)]["name"].value_counts()
d_va = df[df["id"].isin(val_ids)]["name"].value_counts()

check = pd.concat([d_tr, d_va], axis=1, keys=["train", "val"]).fillna(0).astype(int)
check["val_%"] = (check["val"] / (check["train"] + check["val"]) * 100).round(1)
print(check)

In [ ]:
print(len(train_ids), len(val_ids))
print(len(set(train_ids) & set(val_ids)))   # phải bằng 0: không ảnh nào nằm ở cả hai tập

# 7. Cấu trúc YOLO 

In [ ]:
import os, shutil
from pathlib import Path

out = Path("/kaggle/working/dataset")
if out.exists():
    shutil.rmtree(out)

for split in ["train", "val"]:
    (out / "images" / split).mkdir(parents=True)
    (out / "labels" / split).mkdir(parents=True)

def put(ids, split):
    for i in ids:
        os.symlink(root / "train/images" / f"{i}.png", out / "images" / split / f"{i}.png")
        lbl = root / "train/labels" / f"{i}.txt"
        if lbl.exists():
            shutil.copy(lbl, out / "labels" / split / f"{i}.txt")

put(train_ids, "train")
put(val_ids, "val")

print("train:", len(list((out / "images/train").glob("*.png"))),
      "ảnh,", len(list((out / "labels/train").glob("*.txt"))), "label")
print("val  :", len(list((out / "images/val").glob("*.png"))),
      "ảnh,", len(list((out / "labels/val").glob("*.txt"))), "label")

In [ ]:
import yaml

data = {
    "path": str(out),
    "train": "images/train",
    "val": "images/val",
    "nc": len(classes),
    "names": {i: n for i, n in enumerate(classes)},
}
with open(out / "data.yaml", "w", encoding="utf-8") as f:
    yaml.dump(data, f, allow_unicode=True, sort_keys=False)

print(open(out / "data.yaml").read())

In [ ]:
for s in ["train", "val"]:
    ni = len(list((out / f"images/{s}").glob("*.png")))
    nl = len(list((out / f"labels/{s}").glob("*.txt")))
    print(s, ni, nl)

# 8. Huấn luyện YOLO 


In [ ]:
!pip install -q ultralytics #cài Ultralytics

from ultralytics import YOLO

model = YOLO("yolo11n.pt")      # hoặc "yolov8n.pt"; bản n là nhẹ nhất, hợp với edge

results = model.train(
    data="/kaggle/working/dataset/data.yaml",
    imgsz=960,          # ảnh gốc 1672x941 và có box nhỏ, nên dùng lớn hơn 640
    epochs=100,
    batch=8,
    patience=30,        # dừng sớm nếu val không cải thiện
    seed=42,
    flipud=0.0,         # không lật dọc
    degrees=0.0,        # không xoay
    project="/kaggle/working/runs",
    name="exp1",
)

In [ ]:
from ultralytics import YOLO
import pandas as pd

model = YOLO("/kaggle/working/runs/exp1/weights/best.pt")
m = model.val(data="/kaggle/working/dataset/data.yaml", imgsz=960, conf=0.001, iou=0.6)

print("mAP50   :", round(m.box.map50, 3))
print("mAP50-95:", round(m.box.map, 3))
print("P / R   :", round(m.box.mp, 3), "/", round(m.box.mr, 3))

res = pd.DataFrame({
    "class": [classes[i] for i in m.box.ap_class_index],
    "AP50": m.box.ap50,
    "AP50-95": m.box.ap,
}).sort_values("AP50-95")
print(res.round(3))

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

for f in ["results.png", "confusion_matrix_normalized.png", "BoxPR_curve.png"]:
    plt.figure(figsize=(12, 7))
    plt.imshow(Image.open(f"/kaggle/working/runs/exp1/{f}")); plt.axis("off"); plt.show()

In [ ]:
model.predict(source="/kaggle/working/dataset/images/val", imgsz=960, conf=0.25,
              save=True, project="/kaggle/working/runs", name="pred_val", exist_ok=True)

import glob
for p in sorted(glob.glob("/kaggle/working/runs/pred_val/*.png"))[:6]:
    plt.figure(figsize=(12, 7)); plt.imshow(Image.open(p)); plt.axis("off"); plt.show()

# 9. Dự đoán tập test bằng YOLO

In [ ]:
from ultralytics import YOLO
import pandas as pd
from ultralytics import YOLO
model = YOLO("/kaggle/working/runs/exp1/weights/best.pt")


results = model.predict(
    source=str(root / "test/images"),
    imgsz=960,
    conf=0.001,      # ngưỡng thấp nếu chấm theo mAP; nếu chấm theo precision/recall thì dùng 0.25
    iou=0.6,
    device=0,
    stream=True,
    verbose=False,
)

rows = []
for r in results:
    img_id = Path(r.path).stem            # ví dụ img_0001
    W, H = r.orig_shape[1], r.orig_shape[0]
    for b in r.boxes:
        x, y, w, h = b.xywhn[0].tolist()  # chuẩn hoá 0-1, kiểu YOLO
        x1, y1, x2, y2 = b.xyxy[0].tolist()  # pixel
        rows.append([img_id, int(b.cls), classes[int(b.cls)], float(b.conf),
                     x, y, w, h, x1, y1, x2, y2])

pred = pd.DataFrame(rows, columns=["id", "cls", "name", "conf", "x", "y", "w", "h", "x1", "y1", "x2", "y2"])
print(pred.shape, "| số ảnh có dự đoán:", pred["id"].nunique(), "/", len(test_imgs))
print(pred.head())

In [ ]:
det_pred = pred.copy()
print(type(det_pred), det_pred.shape)          # phải là DataFrame, khoảng (N, 12)
print("Ảnh có dự đoán:", det_pred["id"].nunique(), "/ 60")

In [ ]:
det_pred = pred[pred["conf"] >= 0.25].copy()
print(det_pred.shape, "| ảnh còn box:", det_pred["id"].nunique(), "/ 60")
print(det_pred.groupby("id").size().describe())

In [ ]:
print(pred["name"].value_counts())
print(pred.groupby("id").size().describe())
print("Ảnh test không có box nào:", len({p.stem for p in test_imgs} - set(pred["id"])))

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

model.predict(source=str(root / "test/images"), imgsz=960, conf=0.25, save=True,
              project="/kaggle/working/runs", name="pred_test", exist_ok=True, verbose=False)

import glob
for p in sorted(glob.glob("/kaggle/working/runs/pred_test/*.png"))[:4]:
    plt.figure(figsize=(12, 7)); plt.imshow(Image.open(p)); plt.axis("off"); plt.show()

In [ ]:
print(sub.head(10))
print(sub.columns.tolist(), sub.shape)
print(test.head())

# 10. Chuẩn bị dữ liệu intent 

In [ ]:
print(train.shape, test.shape, sub.shape)
print("Cột train:", train.columns.tolist())
print("Cột test :", test.columns.tolist())
print("Cột sub  :", sub.columns.tolist())

display(train.head(10))
display(test.head(5))
display(sub.head(5))

kiểm tra chất lượng dữ liệu 

In [ ]:
print(train.isnull().sum())
print("Dòng trùng:", train.duplicated().sum())
print(train.dtypes)

phân bố nhãn 

In [ ]:
col = "intent"
print(train[col].value_counts())
print("Số nhãn khác nhau:", train[col].nunique())

In [ ]:
text_col = "question"   
print(train[text_col].str.len().describe())
print(train.sample(5, random_state=0)[[text_col, col]])

In [ ]:
print(sub["task_type"].value_counts())
print(sub.head(5))
print(sub.tail(5))

# 11. Chia train/validation 

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

col, text_col = "intent", "question"

vc = train[col].value_counts()
print(vc)
print("Số intent:", len(vc), "| ít nhất:", vc.min())

use_strat = vc.min() >= 2 and len(vc) <= int(len(train) * 0.2)   # val phải đủ chỗ cho mỗi intent
tr_df, va_df = train_test_split(
    train, test_size=0.2, random_state=42,
    stratify=train[col] if use_strat else None,
)
tr_df, va_df = tr_df.reset_index(drop=True), va_df.reset_index(drop=True)

print("Stratify:", use_strat)
print("Train:", len(tr_df), "| Val:", len(va_df))

check = pd.concat([tr_df[col].value_counts(), va_df[col].value_counts()],
                  axis=1, keys=["train", "val"]).fillna(0).astype(int)
print(check)
print("Intent không có ở val:", list(check[check["val"] == 0].index))

In [ ]:
pd.set_option("display.max_colwidth", 120)
print(train.groupby("intent").head(3).sort_values("intent")[["question_id", "question", "intent"]])
print(test.head(10))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, accuracy_score

clf = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), analyzer="char_wb", lowercase=True),
    LogisticRegression(max_iter=1000, class_weight="balanced"),
)
clf.fit(tr_df["question"], tr_df["intent"])

pred_va = clf.predict(va_df["question"])
print("Accuracy:", accuracy_score(va_df["intent"], pred_va))
print(classification_report(va_df["intent"], pred_va, zero_division=0))

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)   # 3 vì OUT_OF_SCOPE chỉ có 3 câu
print(cross_val_score(clf, train["question"], train["intent"], cv=cv).mean())

# 12. Tạo vocabulary 

In [ ]:
import re
from collections import Counter

def tokenize(s):
    s = s.lower().strip()
    return re.findall(r"\w+", s, flags=re.UNICODE)    # giữ chữ có dấu tiếng Việt

counter = Counter()
for q in tr_df["question"]:
    counter.update(tokenize(q))

print("Số từ khác nhau:", len(counter))
print(counter.most_common(20))

In [ ]:
PAD, UNK = "<pad>", "<unk>"

min_freq = 1            # dữ liệu rất nhỏ nên giữ mọi từ; tăng lên 2 nếu có nhiều từ hiếm
itos = [PAD, UNK] + [w for w, c in counter.most_common() if c >= min_freq]
stoi = {w: i for i, w in enumerate(itos)}

print("Kích thước vocab:", len(itos))

In [ ]:
import numpy as np

def encode(s, max_len):
    ids = [stoi.get(w, stoi[UNK]) for w in tokenize(s)][:max_len]
    return ids + [stoi[PAD]] * (max_len - len(ids))

lens = tr_df["question"].map(lambda s: len(tokenize(s)))
print(lens.describe())
max_len = int(lens.max())          # hoặc percentile 95 nếu có câu rất dài

X_tr = np.array([encode(q, max_len) for q in tr_df["question"]])
X_va = np.array([encode(q, max_len) for q in va_df["question"]])

# mã hoá nhãn
intents = sorted(train["intent"].unique())
lab2id = {l: i for i, l in enumerate(intents)}
y_tr = tr_df["intent"].map(lab2id).to_numpy()
y_va = va_df["intent"].map(lab2id).to_numpy()
print(lab2id, X_tr.shape, X_va.shape)

In [ ]:
oov = sum(w not in stoi for q in va_df["question"] for w in tokenize(q))
tot = sum(len(tokenize(q)) for q in va_df["question"])
print(f"Từ ngoài vocab ở val: {oov}/{tot} = {oov/tot:.1%}")

# 13. Định nghĩa TextCNN 

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TextCNN(nn.Module):
    def __init__(self, vocab_size, num_classes, emb_dim=64,
                 kernel_sizes=(2, 3, 4), num_filters=32, dropout=0.5, pad_idx=0):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.convs = nn.ModuleList(
            [nn.Conv1d(emb_dim, num_filters, k, padding=k // 2) for k in kernel_sizes]
        )
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(kernel_sizes), num_classes)

    def forward(self, x):                       # x: (batch, seq_len)
        e = self.emb(x).transpose(1, 2)         # (batch, emb_dim, seq_len)
        feats = [F.relu(conv(e)).max(dim=2).values for conv in self.convs]  # max-pool theo thời gian
        h = self.drop(torch.cat(feats, dim=1))  # (batch, num_filters * số kernel)
        return self.fc(h)                       # logits: (batch, num_classes)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model_txt = TextCNN(
    vocab_size=len(itos),
    num_classes=len(lab2id),
    pad_idx=stoi[PAD],
).to(device)

print(model_txt)
print("Số tham số:", sum(p.numel() for p in model_txt.parameters()))

xb = torch.tensor(X_tr[:4], dtype=torch.long).to(device)
print(model_txt(xb).shape)      # mong đợi: torch.Size([4, 5])

# 14. Huấn luyện TextCNN 

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
np.random.seed(42)

Xtr_t = torch.tensor(X_tr, dtype=torch.long)
ytr_t = torch.tensor(y_tr, dtype=torch.long)
Xva_t = torch.tensor(X_va, dtype=torch.long)
yva_t = torch.tensor(y_va, dtype=torch.long)

train_dl = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=8, shuffle=True)
val_dl   = DataLoader(TensorDataset(Xva_t, yva_t), batch_size=32)

# trọng số lớp: lớp càng hiếm trọng số càng cao
counts = np.bincount(y_tr, minlength=len(lab2id))
weights = counts.sum() / (len(counts) * np.maximum(counts, 1))
criterion = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float).to(device))
print("Trọng số lớp:", dict(zip(lab2id, weights.round(2))))

In [ ]:
import copy

def evaluate(model, dl):
    model.eval()
    correct, total, loss_sum = 0, 0, 0.0
    with torch.no_grad():
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            loss_sum += criterion(out, yb).item() * len(yb)
            correct += (out.argmax(1) == yb).sum().item()
            total += len(yb)
    return loss_sum / total, correct / total

# khởi tạo lại model để chạy lại cell này không bị train tiếp từ lần trước
model_txt = TextCNN(len(itos), len(lab2id), pad_idx=stoi[PAD]).to(device)
optimizer = torch.optim.Adam(model_txt.parameters(), lr=1e-3, weight_decay=1e-4)

EPOCHS, PATIENCE = 100, 20
best_acc, best_loss, best_state, wait = 0.0, 1e9, None, 0
history = []

for ep in range(1, EPOCHS + 1):
    model_txt.train()
    tr_loss, n = 0.0, 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model_txt(xb), yb)
        loss.backward()
        optimizer.step()
        tr_loss += loss.item() * len(yb); n += len(yb)

    va_loss, va_acc = evaluate(model_txt, val_dl)
    history.append((ep, tr_loss / n, va_loss, va_acc))
    if ep % 10 == 0 or ep == 1:
        print(f"ep {ep:3d} | train loss {tr_loss/n:.3f} | val loss {va_loss:.3f} | val acc {va_acc:.3f}")

    # chọn model tốt nhất: acc cao hơn, hoặc acc bằng nhưng loss thấp hơn
    if va_acc > best_acc or (va_acc == best_acc and va_loss < best_loss):
        best_acc, best_loss, wait = va_acc, va_loss, 0
        best_state = copy.deepcopy(model_txt.state_dict())
    else:
        wait += 1
        if wait >= PATIENCE:
            print("Dừng sớm ở epoch", ep); break

model_txt.load_state_dict(best_state)
print(f"Tốt nhất: val acc {best_acc:.3f}, val loss {best_loss:.3f}")

In [ ]:
from sklearn.metrics import classification_report

model_txt.eval()
with torch.no_grad():
    pred = model_txt(Xva_t.to(device)).argmax(1).cpu().numpy()

names = [k for k, _ in sorted(lab2id.items(), key=lambda kv: kv[1])]
print(classification_report(y_va, pred, labels=list(range(len(names))),
                            target_names=names, zero_division=0))

In [ ]:
import json
torch.save(best_state, "/kaggle/working/textcnn.pt")
json.dump({"itos": itos, "lab2id": lab2id, "max_len": max_len},
          open("/kaggle/working/textcnn_meta.json", "w"), ensure_ascii=False)

In [ ]:
import os
for f in ["textcnn.pt", "textcnn_meta.json"]:
    p = f"/kaggle/working/{f}"
    print(f, "OK" if os.path.exists(p) else "CHƯA CÓ", os.path.getsize(p) if os.path.exists(p) else "")

In [ ]:
meta = json.load(open("/kaggle/working/textcnn_meta.json", encoding="utf-8"))
itos, lab2id, max_len = meta["itos"], meta["lab2id"], meta["max_len"]
stoi = {w: i for i, w in enumerate(itos)}

model_txt = TextCNN(len(itos), len(lab2id), pad_idx=stoi["<pad>"]).to(device)
model_txt.load_state_dict(torch.load("/kaggle/working/textcnn.pt", map_location=device))
model_txt.eval()

# 15. Đánh giá TextCNN 

In [ ]:
import numpy as np, torch
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

model_txt.eval()
with torch.no_grad():
    logits = model_txt(Xva_t.to(device))
    prob = torch.softmax(logits, dim=1).cpu().numpy()
pred = prob.argmax(1)

names = [k for k, _ in sorted(lab2id.items(), key=lambda kv: kv[1])]
print("Accuracy :", round(accuracy_score(y_va, pred), 3))
print("Macro-F1 :", round(f1_score(y_va, pred, average="macro", zero_division=0), 3))
print(classification_report(y_va, pred, labels=list(range(len(names))),
                            target_names=names, zero_division=0))

In [ ]:
import pandas as pd
cm = confusion_matrix(y_va, pred, labels=list(range(len(names))))
print(pd.DataFrame(cm, index=[f"thật:{n}" for n in names], columns=[f"đoán:{n}" for n in names]))

In [ ]:
pd.set_option("display.max_colwidth", 120)
wrong = va_df.copy()
wrong["đoán"] = [names[i] for i in pred]
wrong["conf"] = prob.max(1).round(2)
print(wrong[wrong["intent"] != wrong["đoán"]][["question", "intent", "đoán", "conf"]])

In [ ]:
import matplotlib.pyplot as plt
h = pd.DataFrame(history, columns=["ep", "train_loss", "val_loss", "val_acc"])
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
h.plot(x="ep", y=["train_loss", "val_loss"], ax=ax[0], title="Loss")
h.plot(x="ep", y="val_acc", ax=ax[1], title="Val accuracy")
plt.show()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

clf = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), analyzer="char_wb"),
                    LogisticRegression(max_iter=1000, class_weight="balanced"))
clf.fit(tr_df["question"], tr_df["intent"])
print("TF-IDF+LR acc:", round(accuracy_score(va_df["intent"], clf.predict(va_df["question"])), 3))

# 16. Dự đoán intent_test

In [ ]:
import torch, numpy as np

# dùng lại hàm encode, tokenize, stoi, max_len từ bước vocabulary
model_txt.eval()
X_te = torch.tensor([encode(q, max_len) for q in test["question"]], dtype=torch.long).to(device)
with torch.no_grad():
    prob_te = torch.softmax(model_txt(X_te), dim=1).cpu().numpy()

id2lab = {i: l for l, i in lab2id.items()}
test["intent"] = [id2lab[i] for i in prob_te.argmax(1)]
test["conf"] = prob_te.max(1).round(2)

pd.set_option("display.max_colwidth", 100)
print(test[["question_id", "question", "intent", "conf"]])
print(test["intent"].value_counts())

In [ ]:
test["intent"] = clf.predict(test["question"])

In [ ]:
test[["question_id", "intent"]].to_csv("/kaggle/working/intent_pred.csv", index=False)

In [ ]:
print(sub["task_type"].value_counts())
print(sub.head(5))
print(sub.tail(5))

# 17. Xem định dạng submit

In [ ]:
print(sub["task_type"].value_counts())
pd.set_option("display.max_colwidth", 200)
for t in sub["task_type"].unique():
    print("\n==", t)
    print(sub[sub["task_type"] == t].head(5))

In [ ]:
# map question_id -> intent đã dự đoán
intent_map = dict(zip(test["question_id"].astype(str), test["intent"]))

sub_out = sub.copy()
is_intent = sub_out["sample_id"].astype(str).isin(intent_map)   # dòng nào có sample_id trùng question_id
sub_out.loc[is_intent, "prediction"] = sub_out.loc[is_intent, "sample_id"].astype(str).map(intent_map)

print("Số dòng intent đã điền:", is_intent.sum(), "/ 28")
print(sub_out[is_intent].head())

# 18. Tạo submission 

In [ ]:
sub_out.to_csv("/kaggle/working/submission.csv", index=False)

chk = pd.read_csv("/kaggle/working/submission.csv")
print(chk.shape, "| dòng trống prediction:", chk["prediction"].isna().sum())
print(chk.columns.tolist() == sub.columns.tolist())

In [ ]:
print(type(pred))        # phải là pandas DataFrame, không phải numpy.ndarray
print(pred.shape)

In [ ]:
import json

intent_map = dict(zip(test["question_id"].astype(str), test["intent"]))

det_map = {}
for img_id, g in det_pred.groupby("id"):
    items = [{
        "class_id": int(r["cls"]),
        "score": round(float(r["conf"]), 4),
        "bbox": [round(float(r["x1"]), 1), round(float(r["y1"]), 1),
                 round(float(r["x2"]), 1), round(float(r["y2"]), 1)],
    } for _, r in g.iterrows()]
    det_map[img_id] = json.dumps(items, separators=(",", ":"))

def fill(row):
    sid = str(row["sample_id"])
    if row["task_type"] == "DETECTION":
        return det_map.get(sid, "[]")
    return intent_map.get(sid, row["prediction"])

sub_out = sub.copy()
sub_out["prediction"] = sub_out.apply(fill, axis=1)
sub_out.to_csv("/kaggle/working/submission.csv", index=False)

chk = pd.read_csv("/kaggle/working/submission.csv")
print(chk.shape, chk.columns.tolist() == sub.columns.tolist())
print("Ảnh có box:", (chk[chk.task_type == "DETECTION"]["prediction"] != "[]").sum(), "/ 60")
print("Còn giá trị mẫu:", chk["prediction"].str.contains(r'"bbox":\[40.0,60.0,120.0,180.0\]').sum())
print(chk[chk.task_type == "DETECTION"].head(3))

# 19. Check submission


In [ ]:
import json
import pandas as pd

chk = pd.read_csv("/kaggle/working/submission.csv")
ref = pd.read_csv(root / "sample_submission.csv")

# 1) Cấu trúc file
print("Shape       :", chk.shape, "| mẫu:", ref.shape)
print("Cột giống   :", chk.columns.tolist() == ref.columns.tolist())
print("sample_id khớp, đúng thứ tự:", chk["sample_id"].tolist() == ref["sample_id"].tolist())
print("task_type khớp:", chk["task_type"].tolist() == ref["task_type"].tolist())
print("Trùng sample_id:", chk["sample_id"].duplicated().sum())
print("prediction trống:", chk["prediction"].isna().sum())

# 2) Phần INTENT
valid_intents = set(train["intent"].unique())
it = chk[chk["task_type"] == "INTENT"]
print("\nINTENT:", len(it), "dòng | intent lạ:", sorted(set(it["prediction"]) - valid_intents))
print(it["prediction"].value_counts().to_dict())

# 3) Phần DETECTION: parse JSON và kiểm tra từng box
det = chk[chk["task_type"] == "DETECTION"]
errs, n_box, bad_json = [], 0, 0
for _, r in det.iterrows():
    try:
        items = json.loads(r["prediction"])
    except Exception:
        bad_json += 1
        continue
    for b in items:
        n_box += 1
        if set(b) != {"class_id", "score", "bbox"}:
            errs.append((r["sample_id"], "sai khoá", list(b)))
            continue
        if not (0 <= b["class_id"] < len(classes)):
            errs.append((r["sample_id"], "class_id ngoài khoảng", b["class_id"]))
        if not (0 <= b["score"] <= 1):
            errs.append((r["sample_id"], "score ngoài [0,1]", b["score"]))
        x1, y1, x2, y2 = b["bbox"]
        if len(b["bbox"]) != 4 or x2 <= x1 or y2 <= y1:
            errs.append((r["sample_id"], "bbox sai (x2<=x1 hoặc y2<=y1)", b["bbox"]))
        if x1 < 0 or y1 < 0 or x2 > 1672 or y2 > 941:
            errs.append((r["sample_id"], "bbox vượt ảnh 1672x941", b["bbox"]))

print("\nDETECTION:", len(det), "dòng | tổng box:", n_box, "| JSON lỗi:", bad_json)
print("Số lỗi box:", len(errs))
for e in errs[:10]:
    print("  ", e)
print("Còn giá trị mẫu:", chk["prediction"].str.contains(r'"bbox":\[40.0,60.0,120.0,180.0\]').sum())
print("\nMẫu 1 dòng:", det.iloc[0]["prediction"][:200])